# Forklift Audio Candidate Extraction

`data/movie_preprocess` の音声を使って、推論候補となる窓を抽出する専用 Notebook です。異常推論や正常モデル学習は行いません。


このノートブックは、前処理済み `*_audio.wav` から「音が急に目立つ時間窓」を探すための検証用です。音声をフレーム単位特徴へ分解し、さらに1.5秒窓へ集約して、後から動画側の異常候補と突き合わせやすいCSVを保存します。


## 処理の流れ

1. 前処理済み音声 `*_audio.wav` を一覧化し、対象clipを選びます。
2. 音声をフレーム単位特徴へ変換し、saliencyを計算します。
3. 1.5秒窓へ集約し、候補窓CSVと診断図を出力します。


## 0. Packages


In [ ]:
# ------------------------------------------------------------
# セル概要: パッケージ確認
# ------------------------------------------------------------
# - librosa / scipy / matplotlib など、音声候補抽出に必要なライブラリを確認します。
# - 通常は環境構築済みなら実行不要なので、コメントアウトしたまま残しています。
# ------------------------------------------------------------

# %pip install jupyterlab==4.4.1 matplotlib==3.10.1 numpy==2.2.4 pandas==2.2.3 scipy==1.15.2 librosa==0.10.2.post1 soundfile==0.13.1


In [ ]:
# ------------------------------------------------------------
# セル概要: ライブラリ読み込み
# ------------------------------------------------------------
# - Path / numpy / pandas / librosa / scipy.signal などを読み込みます。
# - 以降のセルでは音声波形、スペクトログラム、表形式結果、診断図を扱います。
# ------------------------------------------------------------

from __future__ import annotations

from io import BytesIO
from pathlib import Path
from typing import Any

import importlib
import sys

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from scipy import signal

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src" / "forklift_features").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(f"Could not find src/forklift_features from {PROJECT_ROOT}")

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from forklift_features import vibration as feature_vibration
feature_vibration = importlib.reload(feature_vibration)


In [ ]:
# ------------------------------------------------------------
# セル概要: 音声解析パラメータ
# ------------------------------------------------------------
# - 入力・出力ディレクトリ、サンプリング周波数、STFT / mel の設定をまとめています。
# - 候補窓の長さ、hop、選択閾値もここで変えると後続処理へ一括で反映されます。
# ------------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "data"
MOVIE_PREPROCESS_DIR = DATA_DIR / "movie_preprocess"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "audio_candidate_extraction"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIO_SAMPLE_RATE = 22050
AUDIO_N_FFT = 1024
AUDIO_WIN_LENGTH = 1024
AUDIO_HOP_LENGTH = 256
AUDIO_N_MELS = 48
AUDIO_FMIN_HZ = 20.0
AUDIO_FMAX_HZ = 10000.0
AUDIO_HIGHPASS_CUTOFF_HZ = 20.0
AUDIO_WINDOW_SEC = 1.5
AUDIO_WINDOW_HOP_SEC = 0.5
AUDIO_BAND_LIMITS_HZ = [(20, 200), (200, 1000), (1000, 4000), (4000, 10000)]
AUDIO_SELECTION_QUANTILE = 65.0
AUDIO_SELECTION_MAD_SCALE = 0.35
AUDIO_MIN_SELECTED_WINDOWS = 2

# TARGET_CLIP_STEMS = ["018_後進時、柱に接触"]
TARGET_CLIP_STEMS = ["1001"]
TARGET_CLIP_STEMS = ["1002"]
TARGET_CLIP_STEMS = ["1003"]


## 1. Audio Manifest


In [ ]:
# ------------------------------------------------------------
# セル概要: 音声ファイル一覧の作成
# ------------------------------------------------------------
# - movie_preprocess 配下の `*_audio.wav` を走査し、category / environment / clip_stem を持つ manifest を作ります。
# - TARGET_CLIP_STEMS が指定されている場合は、解析対象をその clip だけに絞ります。
# ------------------------------------------------------------

# 関数メモ: remove_audio_suffix
# - `*_audio.wav` のファイル名から、front/rear/video と共通する clip_stem を取り出します。
# - 期待した命名規則でないファイルは None にして、manifest へ混ぜないようにします。

def remove_audio_suffix(path: Path) -> str | None:
    suffix = "_audio.wav"
    if path.name.endswith(suffix):
        return path.name[: -len(suffix)]
    return None


# 関数メモ: build_audio_manifest
# - 前処理済み音声ファイルを category / environment ごとに走査して一覧表を作ります。
# - 後続セルがファイルシステムを直接触らず、DataFrame だけを見ればよい形に整えます。

def build_audio_manifest(movie_preprocess_dir: Path = MOVIE_PREPROCESS_DIR) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for category_dir in sorted(path for path in movie_preprocess_dir.iterdir() if path.is_dir()):
        category = category_dir.name
        for environment_dir in sorted(path for path in category_dir.iterdir() if path.is_dir()):
            environment = environment_dir.name
            for audio_path in sorted(environment_dir.glob("*_audio.wav")):
                clip_stem = remove_audio_suffix(audio_path)
                if clip_stem is None:
                    continue
                rows.append({
                    "category": category,
                    "environment": environment,
                    "clip_stem": clip_stem,
                    "audio_path": audio_path,
                })
    return pd.DataFrame(rows).sort_values(["category", "environment", "clip_stem"]).reset_index(drop=True)


# 関数メモ: choose_target_audio_clips
# - TARGET_CLIP_STEMS が指定されている場合に解析対象を絞り込みます。
# - 空リストにすれば manifest 全件を対象にできるため、検証範囲をここで調整できます。

def choose_target_audio_clips(audio_manifest_df: pd.DataFrame) -> pd.DataFrame:
    if TARGET_CLIP_STEMS:
        target_df = audio_manifest_df[audio_manifest_df["clip_stem"].isin(TARGET_CLIP_STEMS)].copy()
    else:
        target_df = audio_manifest_df.copy()
    return target_df.sort_values(["category", "environment", "clip_stem"]).reset_index(drop=True)


audio_manifest_df = build_audio_manifest()
target_audio_manifest_df = choose_target_audio_clips(audio_manifest_df)
print("all audio clips:", len(audio_manifest_df))
print("target audio clips:", len(target_audio_manifest_df))
display(target_audio_manifest_df)


## 2. Audio Feature Extraction


In [ ]:
# ------------------------------------------------------------
# セル概要: フレーム単位特徴と候補窓スコア
# ------------------------------------------------------------
# - 音声の前処理、STFT、帯域エネルギー、saliency、窓単位集計を行う関数群です。
# - 最終的には「衝撃音・急変音らしい時間窓」を `is_audio_candidate` として抽出します。
# ------------------------------------------------------------

# 関数メモ: show_figure
# - matplotlib figure を Notebook 上に PNG として表示し、figure を閉じます。
# - 大量の診断図を出す場合でも、未close figure が溜まってメモリを圧迫しにくくします。

def show_figure(figure: plt.Figure) -> None:
    buffer = BytesIO()
    figure.savefig(buffer, format="png", bbox_inches="tight")
    plt.close(figure)
    buffer.seek(0)
    display(Image(data=buffer.getvalue()))


# 関数メモ: load_audio_waveform
# - 音声ファイルを指定サンプリング周波数・モノラルで読み込みます。
# - 以降の特徴量計算が同じ時間解像度で比較できるよう、float32 波形として返します。

def load_audio_waveform(audio_path: Path, sample_rate: int = AUDIO_SAMPLE_RATE) -> tuple[np.ndarray, int]:
    waveform, actual_sample_rate = librosa.load(audio_path, sr=sample_rate, mono=True)
    return waveform.astype(np.float32), int(actual_sample_rate)


# 関数メモ: preprocess_audio_waveform
# - 音声のDC成分を除去し、必要に応じて低域のゆっくりした揺れをハイパスで落とします。
# - 衝撃音候補を拾いやすくするため、平均値や極低周波の影響を抑えます。

def preprocess_audio_waveform(waveform: np.ndarray, sample_rate: int, highpass_cutoff_hz: float = AUDIO_HIGHPASS_CUTOFF_HZ) -> np.ndarray:
    if waveform.size == 0:
        return waveform.astype(np.float32)
    centered = waveform.astype(np.float32) - float(np.mean(waveform))
    if highpass_cutoff_hz <= 0:
        return centered
    nyquist = 0.5 * float(sample_rate)
    normalized_cutoff = min(max(highpass_cutoff_hz / nyquist, 1e-6), 0.99)
    sos = signal.butter(2, normalized_cutoff, btype="highpass", output="sos")
    return signal.sosfiltfilt(sos, centered).astype(np.float32)


# 関数メモ: compute_spectral_flux
# - 隣接フレーム間でスペクトルがどれだけ増えたかを計算します。
# - 急に新しい周波数成分が出る衝撃音・接触音では値が大きくなりやすい特徴です。

def compute_spectral_flux(magnitude_spectrogram: np.ndarray) -> np.ndarray:
    if magnitude_spectrogram.shape[1] == 0:
        return np.zeros((0,), dtype=np.float32)
    frame_diff = np.diff(magnitude_spectrogram, axis=1, prepend=magnitude_spectrogram[:, :1])
    positive_diff = np.maximum(frame_diff, 0.0)
    return np.sqrt(np.sum(positive_diff ** 2, axis=0)).astype(np.float32)


# 関数メモ: compute_band_energy_features
# - 指定した周波数帯ごとに、全エネルギーに対する帯域エネルギー比を作ります。
# - 低域・中域・高域のどこに音の変化が出たかを分けて見られるようにします。

def compute_band_energy_features(power_spectrogram: np.ndarray, sample_rate: int) -> dict[str, np.ndarray]:
    fft_frequencies = librosa.fft_frequencies(sr=sample_rate, n_fft=AUDIO_N_FFT)
    total_energy = np.maximum(np.sum(power_spectrogram, axis=0), 1e-6)
    band_features: dict[str, np.ndarray] = {}
    for low_hz, high_hz in AUDIO_BAND_LIMITS_HZ:
        mask = (fft_frequencies >= float(low_hz)) & (fft_frequencies < float(high_hz))
        band_name = f"band_{int(low_hz)}_{int(high_hz)}"
        if not np.any(mask):
            band_features[band_name] = np.zeros((power_spectrogram.shape[1],), dtype=np.float32)
        else:
            band_energy = np.sum(power_spectrogram[mask, :], axis=0)
            band_features[band_name] = (band_energy / total_energy).astype(np.float32)
    return band_features


# 関数メモ: extract_framewise_audio_features
# - STFT / mel / RMS / ZCR / flatness / spectral flux などをフレーム単位でまとめて計算します。
# - この関数の出力が、saliency と窓スコアを作るための基礎特徴になります。

def extract_framewise_audio_features(waveform: np.ndarray, sample_rate: int) -> dict[str, Any]:
    stft = librosa.stft(
        y=waveform,
        n_fft=AUDIO_N_FFT,
        hop_length=AUDIO_HOP_LENGTH,
        win_length=AUDIO_WIN_LENGTH,
        center=True,
    )
    magnitude = np.abs(stft).astype(np.float32)
    power = (magnitude ** 2).astype(np.float32)
    mel = librosa.feature.melspectrogram(
        S=power,
        sr=sample_rate,
        n_mels=AUDIO_N_MELS,
        fmin=AUDIO_FMIN_HZ,
        fmax=min(AUDIO_FMAX_HZ, 0.5 * float(sample_rate)),
    ).astype(np.float32)
    log_mel = np.log1p(mel).astype(np.float32)
    rms = librosa.feature.rms(y=waveform, frame_length=AUDIO_WIN_LENGTH, hop_length=AUDIO_HOP_LENGTH, center=True)[0].astype(np.float32)
    zcr = librosa.feature.zero_crossing_rate(y=waveform, frame_length=AUDIO_WIN_LENGTH, hop_length=AUDIO_HOP_LENGTH, center=True)[0].astype(np.float32)
    flatness = librosa.feature.spectral_flatness(S=np.maximum(power, 1e-10))[0].astype(np.float32)
    flux = compute_spectral_flux(magnitude)
    band_energies = compute_band_energy_features(power, sample_rate=sample_rate)
    frame_count = log_mel.shape[1]
    frame_times_sec = librosa.frames_to_time(np.arange(frame_count), sr=sample_rate, hop_length=AUDIO_HOP_LENGTH).astype(np.float32)
    scalar_features = {
        "rms": rms[:frame_count],
        "spectral_flux": flux[:frame_count],
        "zcr": zcr[:frame_count],
        "spectral_flatness": flatness[:frame_count],
    }
    for band_name, values in band_energies.items():
        scalar_features[band_name] = values[:frame_count]
    return {
        "frame_times_sec": frame_times_sec,
        "log_mel": log_mel[:, :frame_count],
        "scalar_features": scalar_features,
    }


# 関数メモ: compute_audio_saliency
# - 複数の音声特徴を頑健に正規化して足し合わせ、フレームごとの目立ち度を作ります。
# - 単一特徴だけでなく、エネルギー・変化量・高域成分を合わせて候補らしさを見ます。

def compute_audio_saliency(framewise_features: dict[str, Any]) -> np.ndarray:
    scalar_features = framewise_features["scalar_features"]
    rms_z = feature_vibration.robust_positive_zscore(scalar_features["rms"])
    flux_z = feature_vibration.robust_positive_zscore(scalar_features["spectral_flux"])
    high_z = feature_vibration.robust_positive_zscore(scalar_features.get("band_4000_10000", np.zeros_like(rms_z)))
    mid_z = feature_vibration.robust_positive_zscore(scalar_features.get("band_1000_4000", np.zeros_like(rms_z)))
    return (0.40 * rms_z + 0.35 * flux_z + 0.15 * high_z + 0.10 * mid_z).astype(np.float32)


# 関数メモ: compute_crest_factor
# - 波形のピーク値とRMSの比を計算し、瞬間的な尖り具合を表します。
# - 衝突や接触音のような短いピークを、平均エネルギーとは別軸で拾います。

def compute_crest_factor(waveform_segment: np.ndarray) -> float:
    if waveform_segment.size == 0:
        return 0.0
    rms = float(np.sqrt(np.mean(np.square(waveform_segment))))
    peak = float(np.max(np.abs(waveform_segment)))
    return peak / max(rms, 1e-6)


# 関数メモ: build_audio_window_table
# - フレーム特徴と波形を固定長窓へ集約し、窓ごとの音声候補スコアを作ります。
# - saliency の平均・最大・上位値・crest factor などを1行にまとめます。

def build_audio_window_table(audio_analysis: dict[str, Any], window_sec: float = AUDIO_WINDOW_SEC, hop_sec: float = AUDIO_WINDOW_HOP_SEC) -> pd.DataFrame:
    duration_sec = float(audio_analysis["duration_sec"])
    sample_rate = int(audio_analysis["sample_rate"])
    processed_waveform = audio_analysis["processed_waveform"]
    framewise_features = audio_analysis["framewise_features"]
    frame_times = framewise_features["frame_times_sec"]
    scalar_features = framewise_features["scalar_features"]
    saliency = audio_analysis["saliency"]
    rows: list[dict[str, Any]] = []
    for start_sec in feature_vibration.build_flow_window_starts(duration_sec=duration_sec, window_sec=window_sec, hop_sec=hop_sec):
        end_sec = float(min(start_sec + window_sec, duration_sec))
        start_sec = max(end_sec - window_sec, 0.0)
        frame_mask = (frame_times >= float(start_sec)) & (frame_times < float(end_sec))
        if not np.any(frame_mask):
            nearest_index = int(np.argmin(np.abs(frame_times - float(start_sec))))
            frame_mask = np.zeros_like(frame_times, dtype=bool)
            frame_mask[nearest_index] = True
        start_sample = max(int(np.floor(start_sec * sample_rate)), 0)
        end_sample = min(int(np.ceil(end_sec * sample_rate)), processed_waveform.shape[0])
        waveform_segment = processed_waveform[start_sample:end_sample]
        low_band = scalar_features.get("band_20_200", np.zeros_like(saliency))
        mid_band = scalar_features.get("band_1000_4000", np.zeros_like(saliency))
        high_band = scalar_features.get("band_4000_10000", np.zeros_like(saliency))
        low_band_mean = float(np.mean(low_band[frame_mask]))
        high_band_mean = float(np.mean(high_band[frame_mask]))
        rows.append({
            "window_start_sec": float(start_sec),
            "window_end_sec": float(end_sec),
            "window_center_sec": float(0.5 * (start_sec + end_sec)),
            "rms_p95": float(np.percentile(scalar_features["rms"][frame_mask], 95)),
            "flux_p95": float(np.percentile(scalar_features["spectral_flux"][frame_mask], 95)),
            "flux_max": float(np.max(scalar_features["spectral_flux"][frame_mask])),
            "high_band_mean": high_band_mean,
            "mid_band_mean": float(np.mean(mid_band[frame_mask])),
            "flatness_p95": float(np.percentile(scalar_features["spectral_flatness"][frame_mask], 95)),
            "saliency_p95": float(np.percentile(saliency[frame_mask], 95)),
            "saliency_max": float(np.max(saliency[frame_mask])),
            "crest_factor": compute_crest_factor(waveform_segment),
            "high_low_ratio": high_band_mean / max(low_band_mean, 1e-6),
        })
    audio_window_df = pd.DataFrame(rows)
    score_feature_names = ["saliency_p95", "flux_p95", "flux_max", "crest_factor", "high_band_mean", "rms_p95", "high_low_ratio"]
    for feature_name in score_feature_names:
        z_values = feature_vibration.robust_positive_zscore(audio_window_df[feature_name].to_numpy(dtype=np.float32))
        audio_window_df[f"{feature_name}_z"] = z_values
    audio_window_df["audio_window_score"] = (
        0.25 * audio_window_df["saliency_p95_z"]
        + 0.25 * audio_window_df["flux_p95_z"]
        + 0.15 * audio_window_df["flux_max_z"]
        + 0.15 * audio_window_df["crest_factor_z"]
        + 0.10 * audio_window_df["high_band_mean_z"]
        + 0.05 * audio_window_df["rms_p95_z"]
        + 0.05 * audio_window_df["high_low_ratio_z"]
    ).astype(np.float32)
    return audio_window_df


# 関数メモ: select_audio_candidate_windows
# - 窓スコアの分位点とMADを使って候補窓の閾値を決めます。
# - スコア分布が平坦な場合でも最低候補数を確保し、確認対象がゼロになりにくいようにします。

def select_audio_candidate_windows(audio_window_df: pd.DataFrame) -> tuple[pd.DataFrame, float]:
    candidate_df = audio_window_df.copy()
    if candidate_df.empty:
        candidate_df["is_audio_candidate"] = False
        candidate_df["audio_selection_threshold"] = np.nan
        return candidate_df, float("nan")
    scores = candidate_df["audio_window_score"].to_numpy(dtype=np.float32)
    selected_mask = np.zeros_like(scores, dtype=bool)
    if scores.size == 1 or np.allclose(scores, scores[0]):
        best_index = int(np.argmax(scores))
        selected_mask[best_index] = True
        selection_threshold = float(scores[best_index])
    else:
        score_median = float(np.median(scores))
        score_mad = float(np.median(np.abs(scores - score_median)))
        robust_threshold = score_median + AUDIO_SELECTION_MAD_SCALE * 1.4826 * score_mad
        quantile_threshold = float(np.percentile(scores, AUDIO_SELECTION_QUANTILE))
        selection_threshold = min(quantile_threshold, robust_threshold)
        selected_mask = scores >= selection_threshold
        min_selected_windows = int(min(max(AUDIO_MIN_SELECTED_WINDOWS, 1), scores.size))
        if int(np.sum(selected_mask)) < min_selected_windows:
            top_indices = np.argsort(scores)[-min_selected_windows:]
            selected_mask[top_indices] = True
            selection_threshold = float(min(selection_threshold, np.min(scores[top_indices])))
    candidate_df["is_audio_candidate"] = selected_mask
    candidate_df["audio_selection_threshold"] = float(selection_threshold)
    return candidate_df, float(selection_threshold)


# 関数メモ: analyze_audio_clip
# - 1本の音声ファイルについて、読み込みから前処理、特徴抽出、候補窓選択までを一括実行します。
# - 後続セルが clip 単位の辞書を見るだけで、波形・特徴・窓表・選択結果へアクセスできます。

def analyze_audio_clip(audio_path: Path) -> dict[str, Any]:
    raw_waveform, sample_rate = load_audio_waveform(audio_path)
    processed_waveform = preprocess_audio_waveform(raw_waveform, sample_rate=sample_rate)
    duration_sec = float(processed_waveform.shape[0] / max(sample_rate, 1))
    framewise_features = extract_framewise_audio_features(processed_waveform, sample_rate=sample_rate)
    saliency = compute_audio_saliency(framewise_features)
    waveform_times_sec = np.arange(processed_waveform.shape[0], dtype=np.float32) / max(sample_rate, 1)
    audio_analysis = {
        "sample_rate": sample_rate,
        "duration_sec": duration_sec,
        "raw_waveform": raw_waveform,
        "processed_waveform": processed_waveform,
        "waveform_times_sec": waveform_times_sec,
        "framewise_features": framewise_features,
        "saliency": saliency,
    }
    audio_window_df = build_audio_window_table(audio_analysis)
    audio_window_df, audio_selection_threshold = select_audio_candidate_windows(audio_window_df)
    selected_window_df = audio_window_df[audio_window_df["is_audio_candidate"]].copy()
    audio_analysis.update({
        "audio_window_df": audio_window_df,
        "selected_window_ranges": list(zip(selected_window_df["window_start_sec"].astype(float), selected_window_df["window_end_sec"].astype(float))),
        "audio_selection_threshold": float(audio_selection_threshold),
    })
    return audio_analysis


## 3. Candidate Window Extraction


In [ ]:
# ------------------------------------------------------------
# セル概要: 対象音声への適用とCSV保存
# ------------------------------------------------------------
# - 選択された音声ファイルを順に解析し、全窓スコアと候補窓だけのCSVを出力します。
# - 後段の動画・特徴量検証で使いやすいよう、clip_stem / category / environment を先頭列に付けています。
# ------------------------------------------------------------

audio_analysis_by_clip: dict[str, dict[str, Any]] = {}
audio_window_rows: list[pd.DataFrame] = []

for _, row in target_audio_manifest_df.iterrows():
    clip_stem = str(row["clip_stem"])
    print(f"audio target clip: {clip_stem}")
    audio_analysis = analyze_audio_clip(Path(row["audio_path"]))
    audio_analysis_by_clip[clip_stem] = audio_analysis
    clip_window_df = audio_analysis["audio_window_df"].copy()
    clip_window_df.insert(0, "clip_stem", clip_stem)
    clip_window_df.insert(1, "category", row["category"])
    clip_window_df.insert(2, "environment", row["environment"])
    audio_window_rows.append(clip_window_df)

    selected_window_df = clip_window_df[clip_window_df["is_audio_candidate"]].copy()
    selected_window_text = ", ".join(
        f"{window_row.window_start_sec:.2f}-{window_row.window_end_sec:.2f}s(score={window_row.audio_window_score:.3f})"
        for window_row in selected_window_df.itertuples(index=False)
    ) if not selected_window_df.empty else "none"
    print(
        f"selected audio windows [{clip_stem}] threshold={audio_analysis['audio_selection_threshold']:.3f}: {selected_window_text}"
    )

audio_window_scores_df = pd.concat(audio_window_rows, axis=0, ignore_index=True) if audio_window_rows else pd.DataFrame()
selected_audio_windows_df = audio_window_scores_df[audio_window_scores_df["is_audio_candidate"]].copy() if not audio_window_scores_df.empty else pd.DataFrame()
audio_window_scores_df.to_csv(OUTPUT_DIR / "audio_window_scores.csv", index=False)
selected_audio_windows_df.to_csv(OUTPUT_DIR / "selected_audio_windows.csv", index=False)

if not selected_audio_windows_df.empty:
    display(selected_audio_windows_df[["clip_stem", "category", "environment", "window_start_sec", "window_end_sec", "audio_window_score", "audio_selection_threshold"]])
else:
    print("No audio candidate windows were selected.")


## 4. Audio Visualization


In [ ]:
# ------------------------------------------------------------
# セル概要: 診断グラフ作成
# ------------------------------------------------------------
# - 波形、基本音声特徴、saliency、窓スコア、選択窓をまとめて可視化します。
# - スコアだけでなく「なぜその時間が候補になったか」を目視確認するためのセルです。
# ------------------------------------------------------------

# 関数メモ: plot_audio_diagnostics
# - 音声解析結果を、波形・特徴・saliency・窓スコアの複数段グラフで表示します。
# - 候補窓が本当に目立つ音に対応しているか、人が確認しやすい診断図を作ります。

def plot_audio_diagnostics(audio_analysis: dict[str, Any]) -> plt.Figure:
    framewise_features = audio_analysis["framewise_features"]
    frame_times = framewise_features["frame_times_sec"]
    scalar_features = framewise_features["scalar_features"]
    waveform_times = audio_analysis["waveform_times_sec"]
    processed_waveform = audio_analysis["processed_waveform"]
    saliency = audio_analysis["saliency"]
    audio_window_df = audio_analysis["audio_window_df"]
    selected_window_df = audio_window_df[audio_window_df["is_audio_candidate"]].copy()

    figure, axes = plt.subplots(5, 1, figsize=(16, 15), constrained_layout=True, sharex=False)
    axes[0].plot(waveform_times, processed_waveform, color="tab:gray", linewidth=0.8)
    axes[0].set_title("Waveform")
    axes[0].set_ylabel("Amplitude")
    axes[0].grid(alpha=0.25)

    axes[1].plot(frame_times, scalar_features["rms"], label="RMS", color="tab:blue", linewidth=1.0)
    axes[1].plot(frame_times, scalar_features["spectral_flux"], label="Spectral Flux", color="tab:orange", linewidth=1.0)
    axes[1].set_title("Framewise Energy Features")
    axes[1].set_ylabel("Feature Value")
    axes[1].grid(alpha=0.25)
    axes[1].legend(loc="upper right")

    if "band_1000_4000" in scalar_features:
        axes[2].plot(frame_times, scalar_features["band_1000_4000"], label="Band 1k-4k", color="tab:green", linewidth=1.0)
    if "band_4000_10000" in scalar_features:
        axes[2].plot(frame_times, scalar_features["band_4000_10000"], label="Band 4k-10k", color="tab:red", linewidth=1.0)
    axes[2].plot(frame_times, scalar_features["spectral_flatness"], label="Spectral Flatness", color="tab:purple", linewidth=1.0, alpha=0.8)
    axes[2].set_title("Framewise Spectral Features")
    axes[2].set_ylabel("Feature Value")
    axes[2].grid(alpha=0.25)
    axes[2].legend(loc="upper right")

    axes[3].plot(frame_times, saliency, label="Audio Saliency", color="tab:brown", linewidth=1.2)
    for start_sec, end_sec in audio_analysis["selected_window_ranges"]:
        axes[3].axvspan(start_sec, end_sec, color="tab:orange", alpha=0.2)
    axes[3].set_title("Framewise Saliency with Selected Windows")
    axes[3].set_ylabel("Saliency")
    axes[3].grid(alpha=0.25)
    axes[3].legend(loc="upper right")

    axes[4].plot(audio_window_df["window_center_sec"], audio_window_df["audio_window_score"], marker="o", color="tab:blue", linewidth=1.1, label="Audio Window Score")
    if not selected_window_df.empty:
        axes[4].scatter(selected_window_df["window_center_sec"], selected_window_df["audio_window_score"], color="tab:red", s=48, zorder=3, label="Selected Windows")
    axes[4].axhline(float(audio_analysis["audio_selection_threshold"]), color="black", linestyle="--", linewidth=1.0, label="Selection Threshold")
    for start_sec, end_sec in audio_analysis["selected_window_ranges"]:
        axes[4].axvspan(start_sec, end_sec, color="tab:orange", alpha=0.12)
    axes[4].set_title("Window-Level Audio Score")
    axes[4].set_xlabel("Time [sec]")
    axes[4].set_ylabel("Window Score")
    axes[4].grid(alpha=0.25)
    axes[4].legend(loc="upper right")
    return figure


for _, row in target_audio_manifest_df.iterrows():
    clip_stem = str(row["clip_stem"])
    print(f"audio diagnostic plot: {clip_stem}")
    figure = plot_audio_diagnostics(audio_analysis_by_clip[clip_stem])
    show_figure(figure)
